# Quick-look for simulations

This notebook can be used to have a quicklook at the that product for the simulations.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PlatoSim libraries
from platosim.lightcurve import LightCurve
from platosim.matplotlibrc import setup_notebook
setup_notebook()

---
## Single mode
---

In [ ]:
path = '/lhome/nicholas/software/workdir/cs-binary/test_hdf5'
# filename = f"{path}/000000002/000000002_Ncam3.1_Q5.hdf5"
filename = f"{path}/000000004/000000004_Ncam1.1_Q3.hdf5"
# filename = f"{path}/000000012/000000012_Ncam2.1_Q1.hdf5"

# path = '/lhome/nicholas/software/workdir/cs-planet/test_hdf5'
# filename = f"{path}/000000001/000000001_Ncam1.5_Q1.hdf5"

# path = '/lhome/nicholas/software/workdir/cs-exomoon/test_hdf5'
# filename = f"{path}/000000001/000000001_Ncam1.1_Q1.hdf5"

# Load a single mission quarter light curve
lc = LightCurve(filename, mode="single")
ds = lc.star()
ds

In [ ]:
# Show in the injected variability
dv = lc.varsource()
fig, ax = plt.subplots(1, 1, figsize=(9,4))
ax.plot(dv.time/86400, dv.flux, 'k-', lw=0.5)
ax.set_xlabel('Time [d]')
ax.set_ylabel('Normalised flux')
ax.set_xlim(dv.time.min()/86400, dv.time.max()/86400)
plt.tight_layout();

In [ ]:
# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=False, figsize=(9,5));

### *Test post-processing*

In [ ]:
# df = lc.detrend(model='wotan', segments=True, replace=True, plot=True)

df = lc.detrend(model='lowess', segments=True, replace=False, plot=False)
fig, ax = lc.plot_detrend(df)
fig.savefig(f'{path}/../detrending.png', bbox_inches='tight', dpi=200)

In [ ]:
# Remove outliers (due to photon noise and cosmic ray hits)
df = lc.clip(model='wotan', sigma_lower=6, sigma_upper=3, flux_unit='ppt', plot=True)

In [ ]:
# Stitch the segments (only needed if detrending leaves big jumps)
# df = lc.stitch(column='flux_detrend', method='lowess', gapsize=0.1, segment=5, plot=True)

---
## Multi mode
---

In [ ]:
# path = '/lhome/nicholas/software/workdir/cs-binary/test_vsc/000000001'

# path = '/lhome/nicholas/software/workdir/cs-planet/tests_vsc/000000001'

path = '/lhome/nicholas/software/workdir/cs-exomoon/test_vsc'
star = '000000002'

# In multi mode we parse the entire directory of files
lcs = LightCurve(f'{path}/{star}', mode="multi")

In [ ]:
lcs = LightCurve(f'{path}', mode="multi")

In [ ]:
lcs.unpack()

### *Use multi-mode as single-mode*

In [ ]:
# Fetch files in folder
filenames = lcs.files(suffix='ftr')
filenames[:3]

In [ ]:
# Again one can fetch the first light curve for this star
lc = LightCurve(filenames[0])
lc.data().head()

### *Simulation statistics*

The `.table` files each contain a small overview of the specific simulation. It is much handier to have a single file to search information from, hence, we can merge to one single overview table as follows. It possible to remove the redundant `.table` files during the process using `clean=True`:

In [ ]:
df = lcs.stat_sim_table(ofile=f'{path}/../table/lc_{star}.tab', clean=True)
df.head()

### *View simulations*

In [ ]:
fig, ax = lcs.plot_multi(suffix='ftr', group=1, camera=1, quarter=False, 
                         flux_median=144, alpha=0.1, figsize=(9,5))

In [ ]:
lc = lcs.merge(suffix='ftr', binsize=1/6, flux_group_mean=True, flux_offset=True, 
               ofile=f'{path}/../final/lc_{star}.ftr')

In [ ]:
lc.plot(input_model=True, flux_unit='ppm', median_filter=1, alpha=0.1, figsize=(9,6));

---
## Final data products
---

In [ ]:
path = '/lhome/nicholas/software/workdir/cs-binary/finals'
star = 'mag085/lc_000000001.ftr'

# In multi mode we parse the entire directory of files
lc = LightCurve(f'{path}/{star}', mode="final")

# Introduce gaps from file
inputFileGap = '/lhome/nicholas/software/workdir/mocka/input/instrumentGAP.tab'
lc.gaps(inputFileGap, replace=True, plot=False);

# Show a quarter light curve
lc.plot(flux_unit='ppm', median_filter=1, input_model=True, legend=False, figsize=(9,5));

In [ ]:
path = '/lhome/nicholas/software/workdir/cs-binary/finals'
star = 'mag085/lc_000000090.ftr'

# In multi mode we parse the entire directory of files
lc = LightCurve(f'{path}/{star}', mode="final")

# Introduce gaps from file
inputFileGap = '/lhome/nicholas/software/workdir/mocka/input/instrumentGAP.tab'
lc.gaps(inputFileGap, replace=True, plot=False);

# Show a quarter light curve
lc.plot(flux_unit='ppt', median_filter=1, input_model=False, legend=False, figsize=(9,5));